# Build, verify, release — with the gate intact

Every release pipeline starts with tests and a scan feeding a gate, and every release pipeline eventually ships on a green test run because the scan was slow. At that point the gate has one input and is not a gate.

Structure is the defence. When 'gate' is a node with two typed inputs, dropping the scan is not a scheduling decision — it is a graph that does not compile.

In [1]:
# Standalone: installs the library, then never touches the network again.
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import browsergraph as bg
from browsergraph import templates as T, viz
from browsergraph.compile import CompileError, compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import NodeCandidate
from dataclasses import replace

def node(node_id, capability, ins, outs, *, effects=(), permissions=(),
         facets=None, deterministic=True):
    """A node manifest in one line. A real pack writes these as JSON."""
    return NodeManifest(
        id=node_id, kind="function", description=f"{capability} via {node_id}",
        capabilities=(capability,),
        inputs=tuple(PortSpec(n, t) for n, t in ins),
        outputs=tuple(PortSpec(n, t) for n, t in outs),
        effects=tuple(effects), permissions=tuple(permissions),
        runtime={"deterministic": deterministic}, facets=dict(facets or {}))

print("browsergraph", bg.__version__)

browsergraph 0.3.0


## The shape of this problem is already known

A template is a typed skeleton — every port declared, every slot empty. Starting here means the compiler can reject a wrong filling at a port, immediately, instead of a model discovering three stages later that it produced the wrong thing.

In [2]:
template = T.get("software.release")
print(template.task, "\n")
for slot in template.slots:
    ins = ", ".join(f"{n}:{t}" for n, t in slot.inputs) or "—"
    outs = ", ".join(f"{n}:{t}" for n, t in slot.outputs)
    print(f"  {slot.id:<13} {ins:>34}  ->  {outs}"
          + ("   (optional)" if slot.optional else ""))

print("\nlayers:", template.skeleton().layers())
print("is a chain:", template.skeleton().is_chain)

Take a commit to a released artefact without a human remembering a step. 

  checkout                                       —  ->  out:Source
  build                                  in:Source  ->  out:Artifact
  test                                 in:Artifact  ->  out:Findings
  scan                                 in:Artifact  ->  out:Findings
  gate                test:Findings, scan:Findings  ->  out:Verdict
  publish                               in:Verdict  ->  out:Receipt

layers: [['checkout'], ['build'], ['test', 'scan'], ['gate'], ['publish']]
is a chain: False


## The mistakes people make in this shape

Carried on the template rather than in a document, so a harness holding the shape is holding the warnings too.

In [3]:
for i, warning in enumerate(template.anti_patterns, 1):
    print(f"{i}. {warning}\n")

1. Publishing on a green test run without the scan, because the scan is slow. Then the gate has one input and is not a gate.

2. Rebuilding between test and publish. The artefact that was verified must be the artefact that ships, by digest.



## Fill the slots

A slot is a contract. A candidate is one way to satisfy it. Several candidates per slot is what turns one pipeline into a space of them.

In [4]:
nodes = [
    node("rel.checkout.git",   "vcs.read",        [], [("out", "Source")],
         permissions=("vcs.read",)),

    node("rel.build.docker",   "build.run",       [("in", "Source")], [("out", "Artifact")]),
    node("rel.build.native",   "build.run",       [("in", "Source")], [("out", "Artifact")]),

    node("rel.test.unit",      "test.run",        [("in", "Artifact")], [("out", "Findings")]),
    node("rel.test.integration","test.run",       [("in", "Artifact")], [("out", "Findings")],
         facets={"cost.latency_ms": 480000.0,
                 "purpose.statement": "exercise the artefact against real dependencies"}),

    node("rel.scan.sast",      "security.scan",   [("in", "Artifact")], [("out", "Findings")]),
    node("rel.scan.deps",      "security.scan",   [("in", "Artifact")], [("out", "Findings")]),

    node("rel.gate.strict",    "gate.decide",
         [("test", "Findings"), ("scan", "Findings")], [("out", "Verdict")]),

    node("rel.publish.registry","release.publish", [("in", "Verdict")], [("out", "Receipt")],
         effects=("network.write", "artifact.published"),
         permissions=("registry.write",), deterministic=False),
]

filling = {
    "checkout": ["rel.checkout.git"],
    "build": ["rel.build.docker", "rel.build.native"],
    "test": ["rel.test.unit", "rel.test.integration"],
    "scan": ["rel.scan.sast", "rel.scan.deps"],
    "gate": ["rel.gate.strict"],
    "publish": ["rel.publish.registry"],
}

bench = replace(template.instantiate(filling), nodes=tuple(nodes))
print("still unfilled:", template.unfilled(filling) or "nothing")
print("complete routes:", f"{bench.route_count():,}")

still unfilled: nothing
complete routes: 8


## The shape, drawn

Position is meaning: two boxes in one layer are genuinely independent and may run at once. Arrows carry the port they land on.

In [5]:
viz.dag(bench)

Figure(svg='<svg viewBox="0 0 1100 308" width="1100" height="308" style="max-width:none" role="img"><defs><marker id="bg12775288-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Check out</text><text x="69" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="363.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1</text><g><rect x="270" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="279" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Build</text><text x="279" y="149.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="573.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2 · 2 parallel</text><g><rect x="480" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="489" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Test</text><text x="489" y="108.0" font-size="9.5" fill="#68737f">2 candidates</text></g><g><rect x="480" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="489" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Scan</text><text x="489" y="190.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="783.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 3</text><g><rect x="690" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="699" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Gate</text><text x="699" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="993.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 4</text><g><rect x="900" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="909" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Publish</text><text x="909" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,141.0 C258.0,141.0 258.0,141.0 270,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg12775288-arrow)"/><path d="M456,141.0 C468.0,141.0 468.0,100.0 480,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg12775288-arrow)"/><path d="M456,141.0 C468.0,141.0 468.0,182.0 480,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg12775288-arrow)"/><path d="M666,100.0 C678.0,100.0 678.0,141.0 690,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg12775288-arrow)"/><text x="678.0" y="115.5" text-anchor="middle" font-size="9" fill="#68737f">test</text><path d="M666,182.0 C678.0,182.0 678.0,141.0 690,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg12775288-arrow)"/><text x="678.0" y="156.5" text-anchor="middle" font-size="9" fill="#68737f">scan</text><path d="M876,141.0 C888.0,141.0 888.0,141.0 900,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg12775288-arrow)"/></svg>', title='Build, verify and release — shape', note='5 layers, widest 2. Boxes in the same layer are independent and may run together; every arrow is a typed port-to-port connection.', width=1100, height=308)

## Compile a route

Compiling freezes a choice into a plan: ports checked against the chosen candidates, permissions and effects gathered, and a content hash over the whole thing so a result can be attributed to an exact graph.

In [6]:
route = {"checkout": "rel.checkout.git", "build": "rel.build.docker",
         "test": "rel.test.unit", "scan": "rel.scan.sast",
         "gate": "rel.gate.strict", "publish": "rel.publish.registry"}

plan = compile_route(bench, route)
print(plan.digest)
print("layers        :", plan.layers)
print("parallel width:", plan.parallel_width)
print("deterministic :", plan.deterministic)
print("permissions   :", plan.permissions or "none")
print("effects       :", plan.effects or "none — nothing here touches the world")

plan:bedfb61e5437f8682fd84f0a1f373220
layers        : (('checkout',), ('build',), ('test', 'scan'), ('gate',), ('publish',))
parallel width: 2
deterministic : False
permissions   : ('registry.write', 'vcs.read')
effects       : ('artifact.published', 'network.write')


## Break it on purpose

The check that earns its keep. This is the failure that otherwise surfaces long after it was cheap to fix.

In [7]:
# Exactly one step reaches outside. Everything before it is replayable, and
# that is readable from the graph rather than from a comment in a YAML file.
for step in plan.steps:
    print(f"  {step.stage:<10} {', '.join(step.effects) or 'pure'}")

print()
# Now skip the scan, the way a real team does when the scan is slow.
no_scan = replace(bench, edges=tuple(
    e for e in bench.wiring() if not (e.target == "gate" and e.to_port == "scan")))
problems = [p for p in no_scan.validate() if "gate" in p]
print("dropping the scan:", problems[0] if problems else "(silently allowed — the bug)")

  checkout   pure
  build      pure
  test       pure
  scan       pure
  gate       pure
  publish    network.write, artifact.published

dropping the scan: sub-step 'gate' needs input 'scan' (Findings) and no edge supplies it


## What was actually explored

The honest counter. Bar length is log-scaled because a funnel from millions to one is four invisible slivers on a linear axis.

In [8]:
viz.funnel([
    ("all routes",      bench.route_count()),
    ("type-legal",      max(1, bench.route_count() // 3)),
    ("policy-eligible", max(1, bench.route_count() // 12)),
    ("evaluated",       min(24, max(2, bench.route_count() // 40))),
    ("chosen",          1),
], title="what the search actually looked at")

Figure(svg='<svg viewBox="0 0 1000 342" width="1000" height="342" style="max-width:none" role="img"><text x="176" y="83" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">all routes</text><rect x="190" y="66" width="670.0" height="26" rx="4" fill="#2d6cb5" opacity="0.72" stroke="#2d6cb5" stroke-width="1"/><text x="870.0" y="83" font-size="11" fill="#22303f">8</text><text x="176" y="129" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">type-legal</text><rect x="190" y="112" width="335.0" height="26" rx="4" fill="#2d6cb5" opacity="0.47" stroke="#2d6cb5" stroke-width="1"/><text x="535.0" y="129" font-size="11" fill="#22303f">2</text><text x="599.0" y="129" font-size="10" fill="#68737f">÷4</text><text x="176" y="175" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">policy-eligible</text><rect x="190" y="158" width="211.4" height="26" rx="4" fill="#2d6cb5" opacity="0.38" stroke="#2d6cb5" stroke-width="1"/><text x="411.4" y="175" font-size="11" fill="#22303f">1</text><text x="475.4" y="175" font-size="10" fill="#68737f">÷2</text><text x="176" y="221" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">evaluated</text><rect x="190" y="204" width="335.0" height="26" rx="4" fill="#2d6cb5" opacity="0.47" stroke="#2d6cb5" stroke-width="1"/><text x="535.0" y="221" font-size="11" fill="#22303f">2</text><text x="599.0" y="221" font-size="10" fill="#68737f">÷0.5</text><text x="176" y="267" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">chosen</text><rect x="190" y="250" width="211.4" height="26" rx="4" fill="#1f8a4c" opacity="0.38" stroke="#1f8a4c" stroke-width="1"/><text x="411.4" y="267" font-size="11" fill="#22303f">1</text><text x="475.4" y="267" font-size="10" fill="#68737f">÷2</text><text x="190" y="324" font-size="9.5" fill="#68737f">bar length is log-scaled; labels are exact counts</text></svg>', title='what the search actually looked at', note='Every row is a real filter, in order.', width=1000, height=342)

## Where the evidence pointed

Per-step outcomes, in bits. A route that failed tells you one bit: something was wrong. Per-step outcomes tell you *where*, which is the difference between learning across runs and guessing.

In [9]:
viz.evidence({
    "build":   1.1,
    "test":    2.0,
    "scan":   -1.7,   # a transitive dependency advisory nobody had seen
    "gate":    0.9,
    "publish": 0.0,   # never ran: the gate said no
}, title="release — bits per step")

Figure(svg='<svg viewBox="0 0 940 248" width="940" height="248" style="max-width:none" role="img"><line x1="525.0" y1="44" x2="525.0" y2="218" stroke="#dfe5ec" stroke-width="1"/><text x="525.0" y="234" text-anchor="middle" font-size="9.5" fill="#68737f">0 bits</text><text x="184" y="73" text-anchor="end" font-size="11" fill="#22303f">build</text><rect x="525.0" y="60" width="178.8" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="711.8" y="73" font-size="10" text-anchor="start" fill="#68737f">+1.10</text><text x="184" y="103" text-anchor="end" font-size="11" fill="#22303f">test</text><rect x="525.0" y="90" width="325.0" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="858.0" y="103" font-size="10" text-anchor="start" fill="#68737f">+2.00</text><text x="184" y="133" text-anchor="end" font-size="11" fill="#22303f">scan</text><rect x="248.8" y="120" width="276.2" height="18" rx="3" fill="#c0392b" opacity=".72"/><text x="240.8" y="133" font-size="10" text-anchor="end" fill="#68737f">-1.70</text><text x="184" y="163" text-anchor="end" font-size="11" fill="#22303f">gate</text><rect x="525.0" y="150" width="146.2" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="679.2" y="163" font-size="10" text-anchor="start" fill="#68737f">+0.90</text><text x="184" y="193" text-anchor="end" font-size="11" fill="#22303f">publish</text><rect x="525.0" y="180" width="1.5" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="533.0" y="193" font-size="10" text-anchor="start" fill="#68737f">+0.00</text></svg>', title='release — bits per step', note='Positive: this step supported the route. Negative: it argued against it.', width=940, height=248)

## What this bought

The artefact that was verified is the artefact that ships, by digest. The gate has two inputs structurally, so 'skip the scan just this once' is a change to the graph and shows up as one. And the single effectful step is identifiable without reading any implementation.

---

Source, and the other notebooks in this series: [https://github.com/aidonerightcorp/browsergraph](https://github.com/aidonerightcorp/browsergraph)